In [1]:
"""torch.compile(fullgraph=True) + __tensor_flatten__ + passthrough custom op."""

import torch


_passthrough_ops: set = set()

class ScaledTensor(torch.Tensor):
    @staticmethod
    def __new__(cls, data, scale):
        t = torch.Tensor._make_wrapper_subclass(cls, data.shape, dtype=data.dtype, device=data.device)
        t._data = data
        t._scale = scale
        return t

    def __init__(self, data, scale):
        pass

    def __repr__(self):
        return f"ScaledTensor(shape={self.shape}, scale={self._scale})"

    def __tensor_flatten__(self):
        return ["_data"], {"scale": self._scale}

    @staticmethod
    def __tensor_unflatten__(inner_tensors, metadata, outer_size, outer_stride):
        return ScaledTensor(inner_tensors["_data"], metadata["scale"])

    @classmethod
    def __torch_dispatch__(cls, func, types, args=(), kwargs=None):
        if func in _passthrough_ops:
            return super().__torch_dispatch__(func, types, args, kwargs or {})
        def unwrap(t):
            return t._data if isinstance(t, ScaledTensor) else t
        return func(*torch.utils._pytree.tree_map(unwrap, args),
                    **torch.utils._pytree.tree_map(unwrap, kwargs or {}))


@torch.library.custom_op("mylib::double_v2", mutates_args=[])
def double_v2(x: torch.Tensor) -> torch.Tensor:
    print(f"x._data: {x._data}")
    ptr = x._data.data_ptr()
    return ScaledTensor(x._data * 2, x._scale)

@double_v2.register_fake
def _(x):
    return torch.empty(x.shape, dtype=x.dtype, device=x.device)

_passthrough_ops.add(torch.ops.mylib.double_v2.default)

def fn(x):
    return torch.ops.mylib.double_v2(x)


t = ScaledTensor(torch.randn(4, device="cuda"), scale=0.5)
r = torch.compile(fn, fullgraph=True)(t)


x._data: FakeTensor(..., device='cuda:0', size=(4,))


TorchRuntimeError: Dynamo failed to run FX node with fake tensors: call_function mylib.double_v2(*(ScaledTensor(shape=torch.Size([4]), scale=0.5),), **{}): got RuntimeError("Cannot access data pointer of Tensor (e.g. FakeTensor, FunctionalTensor). If you're using torch.compile/export/fx, it is likely that we are erroneously tracing into a custom kernel. To fix this, please wrap the custom kernel into an opaque custom op. Please see the following for details: https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html")

from user code:
   File "/tmp/ipykernel_548496/2147350943.py", line 52, in fn
    return torch.ops.mylib.double_v2(x)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
